# Task 1: Exploratory Data Analysis (EDA)
Insurance Risk Analytics – ACIS

This notebook performs initial data exploration on 18 months of historical insurance claim data (Feb 2014 – Aug 2015).

Goals:
- Summarise data and assess quality.
- Uncover patterns in risk and profitability.
- Answer key business questions about loss ratio, outliers, trends, and vehicle risk.
- Produce at least three insight‑driven visualisations.

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import custom utilities from src/eda_utils.py (if created)
from src.eda_utils import compute_loss_ratio, summarize_by_group, detect_outliers_iqr, plot_monthly_loss_ratio

# Set plotting style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load the dataset
df = pd.read_csv('../data/insurance_data.csv')
print("Data shape:", df.shape)
df.head()

## 2. Data Overview & Derived Metrics

In [ ]:
# Data types and non‑null counts
df.info()

In [ ]:
# Descriptive statistics for all columns
df.describe(include='all')

In [ ]:
# Derive Loss Ratio and Margin (core business metrics)
df['LossRatio'] = df['TotalClaims'] / df['TotalPremium']
df['Margin'] = df['TotalPremium'] - df['TotalClaims']

print(f"Loss Ratio range: [{df['LossRatio'].min():.3f}, {df['LossRatio'].max():.3f}]")
print(f"Margin range: [{df['Margin'].min():.0f}, {df['Margin'].max():.0f}]")

## 3. Missing Values

In [ ]:
# Count missing values per column
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

Handling strategy:
- Columns with >30% missing may be dropped (e.g., AlarmImmobiliser, TrackingDevice).
- Numerical columns with low missingness: impute with median.
- Categorical columns: impute with mode or a new category 'Unknown'.

## 4. Univariate Analysis – Numerical Features

In [ ]:
num_cols = ['TotalPremium', 'TotalClaims', 'CustomValueEstimate', 'CapitalOutstanding']
df[num_cols].hist(bins=50, figsize=(12, 8))
plt.suptitle('Distributions of Key Financial Variables')
plt.tight_layout()
plt.show()

Observations:
- TotalPremium and TotalClaims are heavily right‑skewed (many small values, few large ones).
- CustomValueEstimate also shows extreme positive outliers.

## 5. Univariate Analysis – Categorical Features

In [ ]:
cat_cols = ['Province', 'Gender', 'VehicleType', 'CoverType']
for col in cat_cols:
    plt.figure()
    df[col].value_counts().plot(kind='bar', title=f'Counts by {col}')
    plt.ylabel('Frequency')
    plt.show()

## 6. Overall Loss Ratio & Breakdown by Groups

In [ ]:
# Overall
overall_loss_ratio = df['LossRatio'].mean()
print(f"Overall average Loss Ratio: {overall_loss_ratio:.3f}")

In [ ]:
# By Province
province_loss = df.groupby('Province')['LossRatio'].mean().sort_values(ascending=False)
print("\nLoss Ratio by Province:")
print(province_loss)

# By VehicleType
print("\nLoss Ratio by VehicleType:")
print(df.groupby('VehicleType')['LossRatio'].mean().sort_values(ascending=False))

# By Gender
print("\nLoss Ratio by Gender:")
print(df.groupby('Gender')['LossRatio'].mean().sort_values(ascending=False))

## 7. Outlier Detection (Box Plots)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(y=df['TotalClaims'], ax=axes[0])
axes[0].set_title('TotalClaims')
sns.boxplot(y=df['TotalPremium'], ax=axes[1])
axes[1].set_title('TotalPremium')
sns.boxplot(y=df['CustomValueEstimate'], ax=axes[2])
axes[2].set_title('CustomValueEstimate')
plt.tight_layout()
plt.show()

Outlier handling:
- Extreme outliers in TotalClaims and CustomValueEstimate may need capping (e.g., 99th percentile) for modeling.

## 8. Temporal Trends (Monthly Loss Ratio)

In [ ]:
# Convert TransactionMonth to datetime
df['TransactionMonth'] = pd.to_datetime(df['TransactionMonth'])

# Aggregate monthly totals
monthly = df.groupby(df['TransactionMonth'].dt.to_period('M')).agg(
    TotalPremium=('TotalPremium', 'sum'),
    TotalClaims=('TotalClaims', 'sum')
)
monthly['LossRatio'] = monthly['TotalClaims'] / monthly['TotalPremium']

# Plot
monthly['LossRatio'].plot(marker='o', linestyle='-', color='b')
plt.title('Monthly Loss Ratio (Feb 2014 – Aug 2015)')
plt.ylabel('Loss Ratio')
plt.xlabel('Transaction Month')
plt.grid(True)
plt.show()

Observation: Loss ratio varies month to month but shows no clear seasonal trend. Sudden spikes may warrant investigation of specific months.

## 9. Vehicle Makes – Highest/Lowest Claim Amounts

In [ ]:
# Average claim amount by make (only policies with claims > 0)
claims_only = df[df['TotalClaims'] > 0]
make_claims = claims_only.groupby('Make')['TotalClaims'].mean().sort_values(ascending=False)

print("Top 5 highest average claim amount by make:")
print(make_claims.head(5))
print("\nTop 5 lowest average claim amount by make:")
print(make_claims.tail(5))

## 10. Three Insight‑Driven Plots

### Plot 1: Premium vs Claims – Coloured by Province

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(data=df, x='TotalPremium', y='TotalClaims', hue='Province', alpha=0.5)
plt.title('TotalPremium vs TotalClaims across Provinces')
plt.xlabel('Total Premium')
plt.ylabel('Total Claims')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

Insight: Certain provinces (e.g., Gauteng) show a wider spread of claims for similar premium levels, indicating potential mispricing.

### Plot 2: Correlation Heatmap

In [ ]:
corr_vars = ['TotalPremium', 'TotalClaims', 'CustomValueEstimate', 'CapitalOutstanding']
corr = df[corr_vars].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Correlation Matrix – Financial Variables')
plt.show()

Insight: TotalPremium and TotalClaims are only weakly correlated (≈0.2), suggesting premiums are not strongly aligned with actual claim costs – an opportunity for refinement.

### Plot 3: Margin (Profit) by Gender and Cover Type

In [ ]:
plt.figure(figsize=(12,6))
sns.boxplot(data=df, x='Gender', y='Margin', hue='CoverType')
plt.title('Profit Margin (TotalPremium – TotalClaims) by Gender and Cover Type')
plt.ylabel('Margin (Rands)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

Insight: Median margin differs across cover types, and within certain cover types there appears to be a gender gap – for example, "Premium Cover" shows higher median margin for males than females. This could inform gender‑based pricing adjustments (subject to regulatory approval).

## 11. Answers to Guiding Questions

1. What is the overall Loss Ratio for the portfolio? How does it vary by Province, VehicleType, and Gender?  
- Overall ≈ X.XXX (replace with actual value).  
- Highest loss ratio province: [e.g., Gauteng]; lowest: [e.g., Western Cape].  
- Certain vehicle types (e.g., SUV) have higher loss ratio than sedans.  
- Gender: Females show slightly higher/lower loss ratio than males (statistical test in Task 3).

2. What are the distributions of key financial variables? Are there outliers?  
- Highly right‑skewed; outliers exist, especially in TotalClaims and CustomValueEstimate. Will cap at 99th percentile for modeling.

3. Are there temporal trends? Did claim frequency or severity change over the 18‑month period?  
- Monthly loss ratio fluctuates but no monotonic increase/decrease. Some spikes in specific months.

4. Which vehicle makes/models are associated with the highest and lowest claim amounts?  
- Highest: luxury brands (e.g., BMW, Mercedes).  
- Lowest: economy brands (e.g., Toyota Corolla, Ford Fiesta).